# Robustness and diagnostic analyses

The paper's section *Robustness and Diagnostic Analyses* checks how some design choices affect the reading of the main results. Each section of this notebook computes one of those diagnostics and writes its values under `data/artifacts/robustness/`.

In [1]:
import os
from utils.dataloader import ROOT
os.chdir(ROOT)  # write data/artifacts/ at the repo root

from pathlib import Path

from utils.detection import load_results
from utils.gha_retrieval import gha_retrieval_report
from utils.prevalence_adjusted_precision import METRICS, prevalence_adjusted_precision_report
from utils.semi_synthetic import semi_synthetic_report

OUTPUT_DIR = Path("data/artifacts/robustness")

## GHA-based candidate retrieval

Can GHA's output replace the manual labels? The dataset pipeline drew every `(workflow, smell)` candidate from GHA's output, and the annotators labeled each one. This section compares the two over every labeled candidate:

- a **flagged** candidate is *confirmed* when its label is `present` (any occurrence count) and *rejected* when its label is `absent`;
- an **unflagged** candidate is *missed* by GHA when its label is `present`.

Inputs, disjoint:
- `data/11_manual_evaluation.csv`: the candidates whose label agrees with GHA. The ground truth is built from these only.
- `data/manual_evaluation_gha_overturned.csv`: the candidates whose label contradicts GHA.

The code stops if a row sits in the wrong file or if a pair appears twice.

Outputs:
- `gha_retrieval_by_smell.csv`: one row per smell;
- `gha_retrieval.json`: the totals, the smells with at least 90% and below 70% confirmation, the lowest and highest rates, and the path and SHA-256 of each input.

In [2]:
by_smell, summary = gha_retrieval_report(OUTPUT_DIR)
flagged, unflagged = summary["flagged"], summary["unflagged"]
print(
    f"Flagged:   {flagged['inspected']} inspected, {flagged['confirmed']} confirmed, "
    f"{flagged['rejected']} rejected ({flagged['confirmation_pct']}% confirmed)"
)
print(
    f"Unflagged: {unflagged['inspected']} inspected, {unflagged['missed']} labeled present "
    f"({unflagged['miss_pct']}% missed by GHA)"
)
for key, sign in [("high_confirmation", ">="), ("low_confirmation", "<")]:
    side = summary[key]
    print(
        f"Confirmation {sign} {side['threshold_pct']:.0f}%: "
        f"{len(side['smells'])}/{summary['n_smells']} smells {side['smells']}"
    )
for key in ("confirmation_range", "miss_range"):
    print(f"{key}: {summary[key]}")
by_smell

Flagged:   1416 inspected, 902 confirmed, 514 rejected (63.7% confirmed)
Unflagged: 1310 inspected, 366 labeled present (27.94% missed by GHA)
Confirmation >= 90%: 9/19 smells ['MissingTokenPermissions', 'MissingForkGuardPR', 'MissingJobTimeout', 'MissingConcurrencyPR', 'MissingForkGuardSchedule', 'MissingConcurrencyGlobal', 'MutableRunnerVersion', 'SuboptimalCacheUsage', 'SingleLangVersionCI']
Confirmation < 70%: 5/19 smells ['MissingPermExternalActions', 'UnsafeArtifactUploadFork', 'UnsafeDeployOnFork', 'MultiCommandStep', 'SingleOSExecution']
confirmation_range: {'min': {'pct': 2.8, 'smells': ['UnsafeDeployOnFork']}, 'max': {'pct': 100.0, 'smells': ['MissingTokenPermissions', 'MutableRunnerVersion', 'SingleLangVersionCI']}}
miss_range: {'min': {'pct': 0.0, 'smells': ['MissingTokenPermissions', 'UnsafeArtifactUploadFork', 'UnsafeDeployOnFork', 'MultiCommandStep']}, 'max': {'pct': 76.0, 'smells': ['MissingJobTimeout']}}


,smell,flagged_inspected,flagged_confirmed,flagged_rejected,confirmation_pct,unflagged_inspected,unflagged_missed,miss_pct
0,MissingPermExternalActions,146,50,96,34.25,53,3,5.66
1,UnpinnedActionVersion,54,43,11,79.63,173,122,70.52
2,MissingTokenPermissions,50,50,0,100.00,50,0,0.00
3,MissingForkGuardPR,54,50,4,92.59,89,39,43.82
4,MissingJobTimeout,51,50,1,98.04,200,152,76.00
5,MissingConcurrencyPR,52,50,2,96.15,51,1,1.96
6,MissingConcurrencyBranch,56,50,6,89.29,52,2,3.85
7,MissingPathFilters,53,39,14,73.58,64,24,37.50
8,MissingForkGuardSchedule,63,60,3,95.24,52,2,3.85
9,UnsafeArtifactUploadFork,63,39,24,61.90,50,0,0.00


## Balanced evaluation

Does the balanced design inflate precision? The n15 set holds 15 present and 15 absent pairs per smell, so its precision assumes that half of the pairs are positive. This section recomputes precision under the prevalence of the candidate pool that the ground truth was sampled from, with the observed recall and false-positive rate of the same rep1 predictions:

- the **candidate pool** is the pool that Step 10 sampled from: the workflows of `data/9_ground_truth_candidates.csv` within the Step 09b LOC cap where GHA reports at most one instance of each study smell. Every pool workflow is a candidate for every smell, flagged by GHA or not;
- the **prevalence** of a smell is GHA's flag rate in the pool times the confirmation rate, plus the unflagged rate times the miss rate. The two rates come from the labels of the previous section, restricted to the candidates in the pool. The **pool prevalence** is the mean over smells: the estimated share of present pairs among the pool's `(workflow, smell)` pairs;
- the **adjusted precision** of a model and design is `pi Re / (pi Re + (1 - pi) FPR)`, with `pi` the pool prevalence and `Re` and `FPR` its recall and false-positive rate on the n15 pairs. Recall does not depend on prevalence, and F1 follows from the adjusted precision and the recall. With a 50% prevalence, the formula gives the precision of the overall detection table.

The code stops if the pool differs from the one that `data/step10_report.json` records, if a pool candidate's `tool_output` contradicts GHA's flag in the pool, or if a model and design does not have as many positive as negative pairs.

Outputs:
- `prevalence_by_smell.csv`: one row per smell, with the counts behind its prevalence;
- `prevalence_adjusted_precision_by_model.csv`: one row per model and design, with the balanced and adjusted precision and F1 and the recall, in percent;
- `prevalence_adjusted_precision.json`: the pool and its prevalence, the design means (the mean over the four models, as in the Avg. row of the overall detection table), the D2 − D1 differences of the means, and the path and SHA-256 of each input file.

In [3]:
prevalence, by_model, summary = prevalence_adjusted_precision_report(OUTPUT_DIR, load_results())
pool = summary["candidate_pool"]
low, high = pool["smell_prevalence_range"]["min"], pool["smell_prevalence_range"]["max"]
print(
    f"Candidate pool: {pool['workflows']} workflows; {pool['labeled_candidates_used']} labeled "
    f"candidates used, {pool['labeled_outside_pool']} outside the pool left out"
)
print(
    f"Pool prevalence: {pool['prevalence_pct']}% (by smell, from {low['pct']}% "
    f"({', '.join(low['smells'])}) to {high['pct']}% ({', '.join(high['smells'])}))"
)
for design, values in summary["designs"].items():
    print(
        f"{design}: precision {values['balanced_precision_pct']}% balanced, "
        f"{values['adjusted_precision_pct']}% adjusted ({values['precision_change_pp']:+.2f} pp); "
        f"recall {values['recall_pct']}%; F1 {values['balanced_f1_pct']}% balanced, "
        f"{values['adjusted_f1_pct']}% adjusted; {values['false_positives']} false positives"
    )
print("D2 - D1 (pp):", summary["d2_minus_d1"])
prevalence

Candidate pool: 14378 workflows; 2685 labeled candidates used, 41 outside the pool left out
Pool prevalence: 29.93% (by smell, from 1.97% (UnsafeArtifactUploadFork) to 93.07% (MissingJobTimeout))
D1: precision 95.38% balanced, 89.92% adjusted (-5.46 pp); recall 79.12%; F1 86.34% balanced, 83.94% adjusted; 45 false positives
D2: precision 93.44% balanced, 85.89% adjusted (-7.55 pp); recall 82.37%; F1 87.55% balanced, 84.08% adjusted; 66 false positives
D2 - D1 (pp): {'balanced_precision_pp': -1.94, 'adjusted_precision_pp': -4.03, 'recall_pp': 3.25, 'balanced_f1_pp': 1.21, 'adjusted_f1_pp': 0.14}


,smell,pool_flagged,pool_unflagged,flag_pct,flagged_inspected,flagged_confirmed,confirmation_pct,unflagged_inspected,unflagged_missed,miss_pct,labeled_outside_pool,prevalence,prevalence_pct
0,MissingPermExternalActions,6336,8042,44.07,121,25,20.66,53,3,5.66,25,0.122708,12.27
1,UnpinnedActionVersion,4987,9391,34.68,54,43,79.63,173,122,70.52,0,0.736798,73.68
2,MissingTokenPermissions,663,13715,4.61,50,50,100.00,50,0,0.00,0,0.046112,4.61
3,MissingForkGuardPR,1998,12380,13.90,54,50,92.59,89,39,43.82,0,0.505977,50.60
4,MissingJobTimeout,11135,3243,77.44,51,50,98.04,200,152,76.00,0,0.930682,93.07
5,MissingConcurrencyPR,7693,6685,53.51,52,50,96.15,51,1,1.96,0,0.523591,52.36
6,MissingConcurrencyBranch,7259,7119,50.49,56,50,89.29,52,2,3.85,0,0.469819,46.98
7,MissingPathFilters,6696,7682,46.57,53,39,73.58,64,24,37.50,0,0.543052,54.31
8,MissingForkGuardSchedule,2371,12007,16.49,63,60,95.24,52,2,3.85,0,0.189171,18.92
9,UnsafeArtifactUploadFork,457,13921,3.18,63,39,61.90,50,0,0.00,0,0.019676,1.97


In [4]:
(100 * by_model.set_index(["design", "model"])[METRICS]).round(2)

balanced_precision  adjusted_precision  recall  \
design model                                                               
D1     GPT-5.4-nano                    96.17               91.48   70.53   
       Gemini 3.5 Flash                98.23               95.95   77.89   
       Claude Haiku 4.5                93.08               85.17   84.91   
       DeepSeek V4 Flash               94.05               87.09   83.16   
D2     GPT-5.4-nano                    92.86               84.74   82.11   
       Gemini 3.5 Flash                94.21               87.43   80.00   
       Claude Haiku 4.5                93.73               86.45   83.86   
       DeepSeek V4 Flash               92.97               84.96   83.51   

                          balanced_f1  adjusted_f1  
design model                                        
D1     GPT-5.4-nano             81.38        79.65  
       Gemini 3.5 Flash         86.89        85.99  
       Claude Haiku 4.5         88.81        85.04  
       DeepSeek V4 Flash        88.27        85.08  
D2     GPT-5.4-nano             87.15        83.40  
       Gemini 3.5 Flash         86.53        83.55  
       Claude Haiku 4.5         88.52        85.14  
       DeepSeek V4 Flash        87.99        84.23

## Semi-synthetic UnsafeDeployOnFork

Do the semi-synthetic positives drive the main results? Natural positives of UnsafeDeployOnFork were scarce, so an LLM mutated real workflows that the annotators had labeled negative for the smell (dataset Step 12d), and the annotators labeled each mutated workflow (Step 12e). This section runs two checks:

- the **GHA sanity check** runs the repository's GHA detector (`modules/gha_detector`) on each base workflow and on its mutated version, with the calls of its batch CLI, and records whether GHA flags UnsafeDeployOnFork. It also looks up each base as a candidate for the smell in the labels of the first section: the bases were selected among the annotated negatives that GHA did not flag, so no base should be flagged; the mutated workflows show whether the mutation adds the construct that GHA's rule looks for;
- the **exclusion sensitivity** recomputes the paper's values without the smell's 30 n15 pairs (15 positive, 10 of them semi-synthetic, and 15 negative), with the loaders and rules of the tables they come from: detection F1, end-to-end Clean Removal and Unnecessary Edit per model and design, and the D1–D2 tests of the paired-tests table, with the Holm family recomputed over the four models. The test effect is the paper's: the D2 − D1 rate difference for detection correctness and clean removal, and the median paired D2 − D1 difference for latency (s) and estimated cost (×1,000 USD). The two binary tests also get their discordant pairs, the only pairs that McNemar's test uses.

The paper reports the F1 and Clean Removal changes and every change of significance. Unnecessary Edit (measured on negatives, all natural for this smell) and the latency and cost effects (which do not depend on labels) stay in the CSVs only.

Inputs:
- `data/12d_synth_UnsafeDeployOnFork_present.json`: the mutated workflows;
- `data/12e_sinth_manual_evaluation.csv`: the annotators' label of each one;
- `data/13_final_ground_truth.json`: the YAML of each base workflow;
- `data/11_manual_evaluation.csv` and `data/manual_evaluation_gha_overturned.csv`: each base as a candidate for the smell;
- `data/ground_truth_n15.json`, the runs and the RQ2 validation files, through the loaders of the tables.

The code stops if a mutated workflow lacks a label or a base YAML, if GHA cannot read a workflow, or if the pairs left out are not the smell's.

Outputs:
- `semi_synthetic_gha_check.csv`: one row per mutated workflow;
- `semi_synthetic_exclusion_rates.csv`: F1, end-to-end Clean Removal and Unnecessary Edit in percent, per model and design, with and without the smell;
- `semi_synthetic_exclusion_tests.csv`: the paired tests per outcome and model, with and without the smell, and the discordant pairs of the binary outcomes;
- `semi_synthetic.json`: the counts of the check, the smell's pairs, the largest change of each rate (percentage points) and of each effect (its own unit), the tests whose significance changes (with their effects and discordant pairs), and the path and SHA-256 of each input file.

In [5]:
gha_check, rates, tests, summary = semi_synthetic_report(OUTPUT_DIR)
check, n15 = summary["gha_check"], summary["gha_check"]["n15"]
print(
    f"Mutated workflows: {check['generated']} generated, {check['accepted']} accepted and "
    f"{check['rejected']} rejected by the annotators"
)
print(
    f"Bases: {check['bases_unflagged_negatives']}/{check['generated']} are annotated negatives "
    f"of GHA's unflagged bucket"
)
print(
    f"GHA flags the smell on {check['bases_flagged']}/{check['generated']} bases and "
    f"{check['mutated_flagged']}/{check['generated']} mutated workflows "
    f"({check['accepted_mutated_flagged']}/{check['accepted']} accepted, "
    f"{check['rejected_mutated_flagged']}/{check['rejected']} rejected)"
)
print(
    f"n15: {n15['semi_synthetic']} semi-synthetic pairs, {n15['mutated_flagged']} flagged; "
    f"{n15['bases_are_n15_negatives']} of their bases are n15 negatives of the smell"
)
exclusion = summary["exclusion"]
smell = exclusion["smell_pairs"]
print(
    f"Exclusion: {exclusion['pairs']['full']} -> {exclusion['pairs']['excluded']} pairs "
    f"({smell['positive']} positive, {smell['semi_synthetic']} of them semi-synthetic, "
    f"and {smell['negative']} negative left out)"
)
for name, top in exclusion["rates_pp"].items():
    print(f"  {name}: changes by at most {top['max_abs_delta']} pp "
          f"({top['model']}, {top['design']})")
for name, top in exclusion["effects"].items():
    print(f"  {name} effect: changes by at most {top['max_abs_delta']} ({top['model']})")
for change in exclusion["significance_changes"]:
    print("Significance change:", change)
tests

Mutated workflows: 49 generated, 45 accepted and 4 rejected by the annotators
Bases: 49/49 are annotated negatives of GHA's unflagged bucket
GHA flags the smell on 0/49 bases and 48/49 mutated workflows (45/45 accepted, 3/4 rejected)
n15: 10 semi-synthetic pairs, 10 flagged; 10 of their bases are n15 negatives of the smell
Exclusion: 570 -> 540 pairs (15 positive, 10 of them semi-synthetic, and 15 negative left out)
  f1: changes by at most 0.3256 pp (gpt54-nano, D1)
  clean_removal: changes by at most 0.7288 pp (haiku45, D1)
  unnecessary_edit: changes by at most 0.3338 pp (deepseek-v4-flash, D2)
  detection effect: changes by at most 0.0033 (gpt54-nano)
  clean_removal effect: changes by at most 0.0074 (gpt54-nano)
  latency_s effect: changes by at most 0.4665 (deepseek-v4-flash)
  cost_1k_usd effect: changes by at most 0.4335 (gemini35-flash)
Significance change: {'outcome': 'clean_removal', 'model': 'gpt54-nano', 'p_holm_full': 0.0258, 'p_holm_excluded': 0.0701, 'effect_full': 0.08

,outcome,model,pairs_full,effect_full,p_holm_full,d1_only_full,d2_only_full,pairs_excluded,effect_excluded,p_holm_excluded,d1_only_excluded,d2_only_excluded,delta,significant_full,significant_excluded
0,detection,gpt54-nano,570,0.040351,1.517482e-02,18,41,540,0.037037,4.181927e-02,18,38,-0.003314,True,True
1,detection,gemini35-flash,570,-0.007018,1.000000e+00,19,15,540,-0.007407,1.000000e+00,19,15,-0.000390,False,False
2,detection,haiku45,570,-0.001754,1.000000e+00,11,10,540,-0.001852,1.000000e+00,11,10,-0.000097,False,False
3,detection,deepseek-v4-flash,570,-0.003509,1.000000e+00,12,10,540,-0.003704,1.000000e+00,11,9,-0.000195,False,False
4,clean_removal,gpt54-nano,255,0.082353,2.582321e-02,17,38,240,0.075000,7.013294e-02,17,35,-0.007353,True,False
5,clean_removal,gemini35-flash,268,0.011194,1.000000e+00,15,18,254,0.007874,1.000000e+00,15,17,-0.003320,False,False
6,clean_removal,haiku45,268,0.000000,1.000000e+00,12,12,254,-0.003937,1.000000e+00,12,11,-0.003937,False,False
7,clean_removal,deepseek-v4-flash,263,0.045627,2.420714e-01,14,26,248,0.044355,2.966152e-01,13,24,-0.001273,False,False
8,latency_s,gpt54-nano,570,-3.506500,1.136752e-48,<NA>,<NA>,540,-3.586500,2.644102e-46,<NA>,<NA>,-0.080000,True,True
9,latency_s,gemini35-flash,570,-3.535000,2.319948e-14,<NA>,<NA>,540,-3.487000,9.858776e-13,<NA>,<NA>,0.048000,True,True


In [6]:
rates.round(2)

,metric,model,design,n_full,pct_full,n_excluded,pct_excluded,delta
0,f1,gpt54-nano,D1,570,81.38,540,81.70,0.33
1,f1,gpt54-nano,D2,570,87.15,540,87.06,-0.09
2,f1,gemini35-flash,D1,570,86.89,540,86.78,-0.11
3,f1,gemini35-flash,D2,570,86.53,540,86.40,-0.13
4,f1,haiku45,D1,570,88.81,540,88.80,-0.00
5,f1,haiku45,D2,570,88.52,540,88.50,-0.02
6,f1,deepseek-v4-flash,D1,570,88.27,540,88.45,0.19
7,f1,deepseek-v4-flash,D2,570,87.99,540,88.16,0.17
8,clean_removal,gpt54-nano,D1,272,64.34,257,64.59,0.25
9,clean_removal,gpt54-nano,D2,263,74.14,248,73.79,-0.35
